# Machine Learning - Programming Assignment
## Comparing Classification models; and deploying an app on Github utilizing Streamlit

**Student Name:** `ADITYA RAJ`  
**Student ID:** `2025AC05657`  
**Date:** `14th August 2026`

In [ ]:
%pip install -r ../requirements.txt

In [2]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(12)
print('Libraries imported successfully.')

Libraries imported successfully.


## Section 1: Dataset Selection and Loading

**Requirements:**
- ≥500 samples
- ≥12features
- Public dataset (UCI/Kaggle)
- Binary/Multi-Class Classification problem

In [3]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

data = pd.read_csv(url, names=columns, sep=',', skipinitialspace=True)

# Dataset information
dataset_name = "Adult Income"
dataset_source = "UCI ML Repository"
n_samples = len(data)  # Total number of rows
n_features = len(data.columns) - 1  # Number of features (excluding target)
problem_type = "binary_classification"

# Problem statement
problem_statement = """
Predicting whether a person earns more than $50K/year or not based on census data. Result is a boolean value (True if >$50K, False otherwise).
This is useful for socioeconomic analysis and resource allocation. Though the creation of a perceptron is more important than the specific dataset, this dataset provides a real-world example of a binary classification problem with a mix of numerical and categorical features.
"""

# Primary evaluation metric
primary_metric = "accuracy"  # not forgertting other metrics like preceision, recall etc., but accuracy is the most straightforward metric for this problem.

# Metric justification
metric_justification = """
Accuracy is chosen here as the class imbalance is mild (~75/25), and it thus provides a straightforward measure of how many predictions are correct out of all predictions made.
In this binary classification problem, it is important to evaluate the overall performance of the model in correctlyclassifying both classes (earning >$50K and <=$50K).
"""

print(f"Dataset: {dataset_name}")
print(f"Source: {dataset_source}")
print(f"Samples: {n_samples}, Features: {n_features}")
print(f"Problem Type: {problem_type}")
print(f"Primary Metric: {primary_metric}")

Dataset: Adult Income
Source: UCI ML Repository
Samples: 32561, Features: 14
Problem Type: binary_classification
Primary Metric: accuracy


## Section 2: Data Preprocessing

In [4]:
data.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [5]:
#From the description on the website: workclass, occupation, native_country have missing values.
print(data['workclass'].unique())
print(data['occupation'].unique())
print(data['native_country'].unique())

<ArrowStringArray>
[       'State-gov', 'Self-emp-not-inc',          'Private',
      'Federal-gov',        'Local-gov',                '?',
     'Self-emp-inc',      'Without-pay',     'Never-worked']
Length: 9, dtype: str
<ArrowStringArray>
[     'Adm-clerical',   'Exec-managerial', 'Handlers-cleaners',
    'Prof-specialty',     'Other-service',             'Sales',
      'Craft-repair',  'Transport-moving',   'Farming-fishing',
 'Machine-op-inspct',      'Tech-support',                 '?',
   'Protective-serv',      'Armed-Forces',   'Priv-house-serv']
Length: 15, dtype: str
<ArrowStringArray>
[             'United-States',                       'Cuba',
                    'Jamaica',                      'India',
                          '?',                     'Mexico',
                      'South',                'Puerto-Rico',
                   'Honduras',                    'England',
                     'Canada',                    'Germany',
                       'Iran'

### Turns out that missing values are represented as '?', so we can replace them with NaN and then drop those rows for simplicity.

In [6]:
data.replace('?', np.nan, inplace=True)
data.dropna(inplace=True)

#also convert the target variable to binary
data['income'] = (data['income'].str.strip() == '>50K').astype(int)

data.describe()

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week,income
count,30162.000000,3.016200e+04,30162.000000,30162.000000,30162.000000,30162.000000,30162.000000
mean,38.437902,1.897938e+05,10.121312,1092.007858,88.372489,40.931238,0.248922
std,13.134665,1.056530e+05,2.549995,7406.346497,404.298370,11.979984,0.432396
min,17.000000,1.376900e+04,1.000000,0.000000,0.000000,1.000000,0.000000
25%,28.000000,1.176272e+05,9.000000,0.000000,0.000000,40.000000,0.000000
50%,37.000000,1.784250e+05,10.000000,0.000000,0.000000,40.000000,0.000000
75%,47.000000,2.376285e+05,13.000000,0.000000,0.000000,45.000000,0.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000,1.000000


In [7]:
data.info()

<class 'pandas.DataFrame'>
Index: 30162 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             30162 non-null  int64
 1   workclass       30162 non-null  str  
 2   fnlwgt          30162 non-null  int64
 3   education       30162 non-null  str  
 4   education_num   30162 non-null  int64
 5   marital_status  30162 non-null  str  
 6   occupation      30162 non-null  str  
 7   relationship    30162 non-null  str  
 8   race            30162 non-null  str  
 9   sex             30162 non-null  str  
 10  capital_gain    30162 non-null  int64
 11  capital_loss    30162 non-null  int64
 12  hours_per_week  30162 non-null  int64
 13  native_country  30162 non-null  str  
 14  income          30162 non-null  int64
dtypes: int64(7), str(8)
memory usage: 5.9 MB


In [9]:
data.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,0
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,0
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,0


In [10]:
#dropping education_num as it is a duplicate of education
data.drop('education_num', axis=1, inplace=True)

#categorical columns encoding
categorical_cols = data.select_dtypes(include='str').columns
le = LabelEncoder()
for col in categorical_cols:
    data[col] = le.fit_transform(data[col])

#splitting the data into train and test sets
X = data.drop('income', axis=1)
y = data['income'].values

X = X.values  # Convert to numpy array for compatibility with scikit-learn

In [11]:
data.head()

,age,workclass,fnlwgt,education,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,5,77516,9,4,0,1,4,1,2174,0,40,38,0
1,50,4,83311,9,2,3,0,4,1,0,0,13,38,0
2,38,2,215646,11,0,5,1,4,1,0,0,40,38,0
3,53,2,234721,1,2,5,0,2,1,0,0,40,38,0
4,28,2,338409,9,2,9,5,2,0,0,0,40,4,0


In [12]:
X

array([[    39,      5,  77516, ...,      0,     40,     38],
       [    50,      4,  83311, ...,      0,     13,     38],
       [    38,      2, 215646, ...,      0,     40,     38],
       ...,
       [    58,      2, 151910, ...,      0,     40,     38],
       [    22,      2, 201490, ...,      0,     20,     38],
       [    52,      3, 287927, ...,      0,     40,     38]],
      shape=(30162, 13))

In [13]:
y

array([0, 0, 0, ..., 0, 0, 1], shape=(30162,))

In [14]:
# Train-test split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=12)

In [15]:
# Feature scaling

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [16]:
# Data-logging
train_samples = len(X_train_scaled)       # Number of training samples
test_samples = len(X_test_scaled)        # Number of test samples
train_test_ratio = train_samples / (train_samples + test_samples)

print(f"Train samples: {train_samples}")
print(f"Test samples: {test_samples}")
print(f"Split ratio: {train_test_ratio:.1%}")

Train samples: 24129
Test samples: 6033
Split ratio: 80.0%
